# Persist daily pairwise statistics

Run the shared package implementation with one compute child at a time. The CLI script uses the same work unit and persistence function. Each date receives a fresh process; the notebook kernel remains the sole synchronous DuckDB writer.

In [ ]:
import sys
from concurrent.futures import ProcessPoolExecutor
from datetime import date, datetime, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

import duckdb

repository = Path("..").resolve()
sys.path.insert(0, str(repository))

from liljegren_analysis import compute_daily_pairwise, persist_daily_pairwise
from scripts.daily_pairwise import DEFAULT_RUNS

In [ ]:
runs = tuple(path.resolve() for path in DEFAULT_RUNS)
database_path = repository / "data" / "daily_pairwise.duckdb"
start_date = date(2025, 1, 1)
end_date = date(2025, 12, 31)
threads_per_worker = 4

if start_date > end_date:
    raise ValueError("start_date must not be after end_date")
if not runs:
    raise ValueError("runs is empty")

dates = [
    start_date + timedelta(days=offset)
    for offset in range((end_date - start_date).days + 1)
]
len(dates)

## Process and persist

The executor has one live child and replaces it after every task. Daily delete-and-insert transactions make reruns resumable and idempotent.

In [ ]:
database_path.parent.mkdir(parents=True, exist_ok=True)
executor = ProcessPoolExecutor(max_workers=1, max_tasks_per_child=1)

try:
    with duckdb.connect(str(database_path)) as writer:
        for completed, result_date in enumerate(dates, start=1):
            processed_date, result, worker_pid, compute_seconds = (
                executor.submit(
                    compute_daily_pairwise,
                    result_date,
                    runs,
                    threads_per_worker,
                ).result()
            )
            if processed_date != result_date:
                raise RuntimeError(
                    f"Expected {result_date}, received {processed_date}"
                )

            persist_daily_pairwise(writer, result_date, result)
            processed_at = datetime.now(ZoneInfo("Asia/Singapore"))
            print(
                f"[{processed_at:%Y-%m-%d %H:%M:%S SGT}] "
                f"Processed {result_date} worker_pid={worker_pid} "
                f"compute={compute_seconds:.1f}s [{completed}/{len(dates)}]"
            )

        writer.execute("CHECKPOINT")
finally:
    executor.shutdown(wait=True, cancel_futures=True)

print(f"Saved daily_pairwise to {database_path}")